# Brain Metastasis Ensemble Pretraining

Self-supervised pretraining for multi-scale ensemble (8, 12, 24, 36 patch sizes).

**Features:**
- Masked reconstruction pretraining on Yale data
- Fully resumable - safe to stop/restart anytime
- Automatic corrupted file tracking
- Progress saved to Drive after each epoch

**Setup:**
1. Run cells 1-5 in order (setup)
2. Run cell 6 to start/resume pretraining

**If session disconnects:** Run cells 1-5, then cell 6 - it will auto-resume.

**Compute usage:** ~2-4 units/hour on T4, ~4-6 units/hour on A100

In [ ]:
# 1. Check GPU & Mount Drive
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/brainMetShare'
DRIVE_DIR = '/content/drive/MyDrive/BrainMetShare'
LOCAL_DATA = '/content/data'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/model", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/model/training_states", exist_ok=True)
print("\nDirectories created!")

In [ ]:
# 2. Install Dependencies
!pip install nibabel scipy tqdm monai -q
print("Dependencies installed!")

In [ ]:
# 3. Upload Code
from google.colab import files
import zipfile

print("Upload brainMetShare_code.zip")
uploaded = files.upload()

# Extract
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall('/content/brainMetShare')
        print(f"Extracted {filename}")

# Verify
import os
src_path = '/content/brainMetShare/src/segmentation'
if os.path.exists(src_path):
    print(f"Code ready: {os.listdir(src_path)}")
else:
    print("WARNING: src/segmentation not found - check zip structure")

In [ ]:
# 4. Copy Pretraining Data to Local SSD
# This is MUCH faster than reading from Drive during training
import time
import shutil

start = time.time()
!df -h /content

# Create local data directory FIRST
os.makedirs(LOCAL_DATA, exist_ok=True)

# Check available data sources
SUPERSET_DIR = f"{DRIVE_DIR}/Superset"
PRETRAINING_SRC = f"{SUPERSET_DIR}/pretraining"
TRAIN_SRC = f"{SUPERSET_DIR}/full/train"

# Alternative: use preprocessed_256 if Superset not available
ALT_TRAIN_SRC = f"{DRIVE_DIR}/preprocessed_256/train"

print("Checking data sources...")
print(f"  Superset pretraining: {os.path.exists(PRETRAINING_SRC)}")
print(f"  Superset train: {os.path.exists(TRAIN_SRC)}")
print(f"  Alt train (preprocessed_256): {os.path.exists(ALT_TRAIN_SRC)}")

# Copy pretraining data
LOCAL_PRETRAIN = f"{LOCAL_DATA}/pretraining"
LOCAL_TRAIN = f"{LOCAL_DATA}/train"

if os.path.exists(PRETRAINING_SRC):
    print("\nCopying pretraining data (this may take a while)...")
    !cp -r "{PRETRAINING_SRC}" "{LOCAL_PRETRAIN}"
    n_cases = len([d for d in os.listdir(LOCAL_PRETRAIN) if os.path.isdir(f"{LOCAL_PRETRAIN}/{d}")])
    print(f"Pretraining: {n_cases} cases")
else:
    print("\nNo dedicated pretraining dir - will use train data for pretraining")
    LOCAL_PRETRAIN = None

# Copy train data (for fine-tuning later, and as fallback for pretraining)
train_src = TRAIN_SRC if os.path.exists(TRAIN_SRC) else ALT_TRAIN_SRC
if os.path.exists(train_src):
    print(f"\nCopying train data from {train_src}...")
    !cp -r "{train_src}" "{LOCAL_TRAIN}"
    n_train = len([d for d in os.listdir(LOCAL_TRAIN) if os.path.isdir(f"{LOCAL_TRAIN}/{d}")])
    print(f"Train: {n_train} cases")
else:
    print("WARNING: No train data found!")

# If no dedicated pretraining, use train
if LOCAL_PRETRAIN is None:
    LOCAL_PRETRAIN = LOCAL_TRAIN
    print(f"Using train data for pretraining: {LOCAL_PRETRAIN}")

print(f"\nData copied in {(time.time()-start)/60:.1f} min")
!df -h /content

In [ ]:
# 5. Setup Python Path & Restore Previous State
import sys
sys.path.insert(0, '/content/brainMetShare/src')
os.chdir(PROJECT_DIR)

# Restore training states from Drive if they exist
DRIVE_STATES = f"{DRIVE_DIR}/model/training_states"
LOCAL_STATES = f"{PROJECT_DIR}/model/training_states"

if os.path.exists(DRIVE_STATES):
    print("Restoring training states from Drive...")
    for f in os.listdir(DRIVE_STATES):
        src = f"{DRIVE_STATES}/{f}"
        dst = f"{LOCAL_STATES}/{f}"
        if os.path.isfile(src):
            shutil.copy(src, dst)
            print(f"  Restored: {f}")

# Restore best models from Drive
DRIVE_MODELS = f"{DRIVE_DIR}/model"
LOCAL_MODELS = f"{PROJECT_DIR}/model"

if os.path.exists(DRIVE_MODELS):
    print("\nRestoring model checkpoints from Drive...")
    for f in os.listdir(DRIVE_MODELS):
        if f.endswith('.pth'):
            src = f"{DRIVE_MODELS}/{f}"
            dst = f"{LOCAL_MODELS}/{f}"
            if os.path.isfile(src):
                shutil.copy(src, dst)
                print(f"  Restored: {f}")

print("\nReady for pretraining!")

In [ ]:
# 6. Pretrain Ensemble (Resumable)
import gc
import json
import shutil
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import numpy as np
import nibabel as nib
from tqdm import tqdm

from segmentation.unet import LightweightUNet3D

# =============================================================================
# CONFIGURATION
# =============================================================================
ENSEMBLE_CONFIGS = {
    'exp1_8patch': {
        'patch_size': 8,
        'base_channels': 20,
        'use_attention': True,
        'use_residual': True,
        'pretrain_epochs': 30,
    },
    'exp3_12patch_maxfn': {
        'patch_size': 12,
        'base_channels': 20,
        'use_attention': True,
        'use_residual': True,
        'pretrain_epochs': 30,
    },
    'improved_24patch': {
        'patch_size': 24,
        'base_channels': 20,
        'use_attention': True,
        'use_residual': True,
        'pretrain_epochs': 30,
    },
    'improved_36patch': {
        'patch_size': 36,
        'base_channels': 20,
        'use_attention': True,
        'use_residual': True,
        'pretrain_epochs': 30,
    },
}

SEQUENCES = ['t1_pre', 't1_gd', 'flair', 't2']
BATCH_SIZE = 4  # Adjust based on GPU memory
LR = 0.001

# Paths
PRETRAIN_DATA = LOCAL_PRETRAIN
MODEL_DIR = Path(f"{PROJECT_DIR}/model")
STATE_DIR = MODEL_DIR / "training_states"
DRIVE_MODEL_DIR = Path(f"{DRIVE_DIR}/model")
DRIVE_STATE_DIR = DRIVE_MODEL_DIR / "training_states"

DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_STATE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# =============================================================================
# CORRUPTED FILE TRACKER
# =============================================================================
class CorruptedFileTracker:
    def __init__(self, cache_path):
        self.cache_path = Path(cache_path)
        self.corrupted_files = set()
        self.load()

    def load(self):
        if self.cache_path.exists():
            try:
                with open(self.cache_path, 'r') as f:
                    self.corrupted_files = set(json.load(f))
                print(f"Loaded {len(self.corrupted_files)} known corrupted files")
            except:
                pass

    def save(self):
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.cache_path, 'w') as f:
            json.dump(list(self.corrupted_files), f)

    def add(self, filepath):
        self.corrupted_files.add(str(filepath))
        self.save()

    def is_corrupted(self, filepath):
        return str(filepath) in self.corrupted_files

corrupted_tracker = CorruptedFileTracker(MODEL_DIR / 'corrupted_files.json')

# =============================================================================
# PRETRAINING DATASET
# =============================================================================
class PretrainingDataset(Dataset):
    def __init__(self, data_dir, sequences, patch_size, target_size=(128, 128, 128), min_modalities=3):
        self.data_dir = Path(data_dir)
        self.sequences = sequences
        self.patch_size = patch_size
        self.target_size = target_size

        # Find valid cases
        self.cases = []
        all_dirs = sorted([d for d in self.data_dir.iterdir() if d.is_dir()])

        print(f"Scanning {len(all_dirs)} directories...")
        for case_dir in tqdm(all_dirs, desc="Validating"):
            count = 0
            for seq in sequences:
                path = case_dir / f"{seq}.nii.gz"
                if path.exists() and not corrupted_tracker.is_corrupted(path):
                    count += 1
            if count >= min_modalities:
                self.cases.append(case_dir)

        print(f"Found {len(self.cases)} valid cases")

    def _load_nifti(self, path):
        try:
            return nib.load(str(path)).get_fdata().astype(np.float32)
        except Exception as e:
            corrupted_tracker.add(path)
            return None

    def _normalize(self, img):
        mean, std = img.mean(), img.std()
        return (img - mean) / std if std > 0 else img

    def _resize(self, img):
        from scipy.ndimage import zoom
        factors = [t / c for t, c in zip(self.target_size, img.shape)]
        return zoom(img, factors, order=1, mode='nearest')

    def _extract_patch(self, images):
        C, H, W, D = images.shape
        p = self.patch_size

        if H < p or W < p or D < p:
            images = np.pad(images, ((0,0), (0,max(0,p-H)), (0,max(0,p-W)), (0,max(0,p-D))), mode='constant')
            C, H, W, D = images.shape

        h = np.random.randint(0, max(1, H-p+1))
        w = np.random.randint(0, max(1, W-p+1))
        d = np.random.randint(0, max(1, D-p+1))

        return images[:, h:h+p, w:w+p, d:d+p]

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        case_dir = self.cases[idx]
        images = []

        for seq in self.sequences:
            path = case_dir / f"{seq}.nii.gz"
            if path.exists() and not corrupted_tracker.is_corrupted(path):
                img = self._load_nifti(path)
                if img is not None:
                    img = self._resize(img)
                    img = self._normalize(img)
                    images.append(img)
                else:
                    images.append(np.zeros(self.target_size, dtype=np.float32))
            else:
                images.append(np.zeros(self.target_size, dtype=np.float32))

        images = np.stack(images, axis=0)
        images = self._extract_patch(images)
        return torch.from_numpy(images).float()

# =============================================================================
# MASKED RECONSTRUCTION
# =============================================================================
class MaskedReconstruction(nn.Module):
    def __init__(self, mask_ratio=0.4, patch_mask_size=4):
        super().__init__()
        self.mask_ratio = mask_ratio
        self.patch_mask_size = patch_mask_size

    def create_mask(self, shape, device):
        B, C, H, W, D = shape
        p = self.patch_mask_size
        nh, nw, nd = max(1, H//p), max(1, W//p), max(1, D//p)
        n_patches = nh * nw * nd
        n_mask = max(1, int(n_patches * self.mask_ratio))

        masks = []
        for _ in range(B):
            mask_idx = np.random.choice(n_patches, min(n_mask, n_patches), replace=False)
            mask = torch.ones(H, W, D, device=device)
            for idx in mask_idx:
                i = (idx // (nw * nd)) * p
                j = ((idx % (nw * nd)) // nd) * p
                k = (idx % nd) * p
                mask[i:min(i+p,H), j:min(j+p,W), k:min(k+p,D)] = 0
            masks.append(mask)

        return torch.stack(masks).unsqueeze(1).expand(-1, C, -1, -1, -1)

    def forward(self, model, images, criterion):
        mask = self.create_mask(images.shape, images.device)
        masked_images = images * mask
        reconstructed = model(masked_images)
        inv_mask = 1 - mask[:, :1, :, :, :]
        return criterion(reconstructed * inv_mask, images[:, :1, :, :, :] * inv_mask)

# =============================================================================
# TRAINING STATE MANAGER
# =============================================================================
class StateManager:
    def __init__(self, local_dir, drive_dir):
        self.local_dir = Path(local_dir)
        self.drive_dir = Path(drive_dir)

    def get_path(self, model_name):
        return self.local_dir / f'{model_name}_pretrain_state.pth'

    def save(self, model_name, epoch, model, optimizer, scheduler, scaler, best_loss, history):
        state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_loss': best_loss,
            'history': history,
            'timestamp': datetime.now().isoformat()
        }

        local_path = self.get_path(model_name)
        torch.save(state, local_path)

        # Also save to Drive
        drive_path = self.drive_dir / f'{model_name}_pretrain_state.pth'
        shutil.copy(local_path, drive_path)

    def load(self, model_name, model, optimizer, scheduler, scaler, device):
        local_path = self.get_path(model_name)
        if not local_path.exists():
            return None

        print(f"Loading state from {local_path}")
        state = torch.load(local_path, map_location=device, weights_only=False)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        scheduler.load_state_dict(state['scheduler_state_dict'])
        scaler.load_state_dict(state['scaler_state_dict'])
        print(f"Resumed from epoch {state['epoch']}, best_loss {state['best_loss']:.4f}")
        return state

    def is_complete(self, model_name):
        return (self.local_dir / f'{model_name}_pretrain_DONE').exists()

    def mark_complete(self, model_name):
        (self.local_dir / f'{model_name}_pretrain_DONE').touch()
        (self.drive_dir / f'{model_name}_pretrain_DONE').touch()

state_manager = StateManager(STATE_DIR, DRIVE_STATE_DIR)

# =============================================================================
# PRETRAIN ONE MODEL
# =============================================================================
def pretrain_model(model_name, config):
    patch_size = config['patch_size']
    epochs = config['pretrain_epochs']

    print(f"\n{'='*60}")
    print(f"PRETRAINING: {model_name} (patch_size={patch_size})")
    print(f"{'='*60}")

    # Check if already complete
    if state_manager.is_complete(model_name):
        print(f"Already complete - skipping")
        return MODEL_DIR / f'{model_name}_pretrained.pth'

    # Create model
    model = LightweightUNet3D(
        in_channels=4,
        out_channels=1,
        base_channels=config['base_channels'],
        use_attention=config['use_attention'],
        use_residual=config['use_residual']
    ).to(device)

    # Dataset
    dataset = PretrainingDataset(
        data_dir=PRETRAIN_DATA,
        sequences=SEQUENCES,
        patch_size=patch_size,
        target_size=(128, 128, 128),
        min_modalities=3
    )

    if len(dataset) == 0:
        print("ERROR: No valid cases!")
        return None

    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)

    print(f"Training on {len(dataset)} cases, {len(loader)} batches/epoch")

    # Pretraining task
    masked_recon = MaskedReconstruction(mask_ratio=0.4, patch_mask_size=max(2, patch_size//4))
    criterion = nn.MSELoss()

    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler('cuda')

    # Try to resume
    start_epoch = 1
    best_loss = float('inf')
    history = {'loss': [], 'lr': []}

    state = state_manager.load(model_name, model, optimizer, scheduler, scaler, device)
    if state:
        start_epoch = state['epoch'] + 1
        best_loss = state['best_loss']
        history = state.get('history', history)

        if start_epoch > epochs:
            print(f"Training already complete")
            state_manager.mark_complete(model_name)
            return MODEL_DIR / f'{model_name}_pretrained.pth'

    checkpoint_path = MODEL_DIR / f'{model_name}_pretrained.pth'

    # Training loop
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        total_loss = 0
        n_batches = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch}/{epochs}")
        for images in pbar:
            images = images.to(device)

            optimizer.zero_grad()

            with autocast('cuda'):
                loss = masked_recon(model, images, criterion)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        scheduler.step()
        avg_loss = total_loss / n_batches
        current_lr = scheduler.get_last_lr()[0]

        history['loss'].append(avg_loss)
        history['lr'].append(current_lr)

        print(f"Epoch {epoch}: Loss={avg_loss:.4f}, LR={current_lr:.6f}")

        # Save best
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'loss': best_loss,
                'config': config
            }, checkpoint_path)
            # Also save to Drive
            shutil.copy(checkpoint_path, DRIVE_MODEL_DIR / f'{model_name}_pretrained.pth')
            print(f"  Saved best model (loss={best_loss:.4f})")

        # Save state (for resume)
        state_manager.save(model_name, epoch, model, optimizer, scheduler, scaler, best_loss, history)
        print(f"  [Checkpoint saved - safe to stop]")

        if epoch % 5 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    state_manager.mark_complete(model_name)
    print(f"\nPretraining complete! Best loss: {best_loss:.4f}")
    return checkpoint_path

# =============================================================================
# RUN PRETRAINING
# =============================================================================
print("\n" + "="*60)
print("ENSEMBLE PRETRAINING")
print("="*60)
print(f"\nModels to pretrain: {list(ENSEMBLE_CONFIGS.keys())}")
print(f"Data directory: {PRETRAIN_DATA}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LR}")

results = {}
for model_name, config in ENSEMBLE_CONFIGS.items():
    checkpoint = pretrain_model(model_name, config)
    results[model_name] = str(checkpoint) if checkpoint else None

    gc.collect()
    torch.cuda.empty_cache()

# Summary
print("\n" + "="*60)
print("PRETRAINING COMPLETE")
print("="*60)
for model_name, path in results.items():
    status = "Done" if path else "Failed"
    print(f"  {model_name}: {status}")
print(f"\nModels saved to: {DRIVE_MODEL_DIR}")

In [ ]:
# 7. Check Progress
import json
from pathlib import Path

print("Training Status:")
print("="*50)

for model_name in ENSEMBLE_CONFIGS.keys():
    state_file = STATE_DIR / f'{model_name}_pretrain_state.pth'
    done_file = STATE_DIR / f'{model_name}_pretrain_DONE'

    if done_file.exists():
        print(f"{model_name}: COMPLETE")
    elif state_file.exists():
        state = torch.load(state_file, map_location='cpu', weights_only=False)
        epochs_total = ENSEMBLE_CONFIGS[model_name]['pretrain_epochs']
        print(f"{model_name}: Epoch {state['epoch']}/{epochs_total}, Loss={state['best_loss']:.4f}")
    else:
        print(f"{model_name}: Not started")

In [ ]:
# 8. Plot Training Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, model_name in enumerate(ENSEMBLE_CONFIGS.keys()):
    ax = axes[idx]
    state_file = STATE_DIR / f'{model_name}_pretrain_state.pth'

    if state_file.exists():
        state = torch.load(state_file, map_location='cpu', weights_only=False)
        history = state.get('history', {})

        if 'loss' in history and len(history['loss']) > 0:
            ax.plot(history['loss'], 'b-', label='Loss')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.legend()
            ax.set_title(f"{model_name} (best={state['best_loss']:.4f})")
        else:
            ax.text(0.5, 0.5, 'No history', ha='center', va='center')
            ax.set_title(model_name)
    else:
        ax.text(0.5, 0.5, 'Not started', ha='center', va='center')
        ax.set_title(model_name)

plt.tight_layout()
plt.savefig(f"{DRIVE_MODEL_DIR}/pretrain_curves.png", dpi=150)
plt.show()
print(f"Saved to {DRIVE_MODEL_DIR}/pretrain_curves.png")

In [ ]:
# 9. Final Save to Drive
import shutil
from datetime import datetime

print("Saving all models and states to Drive...")

# Save models
for f in MODEL_DIR.glob('*.pth'):
    dst = DRIVE_MODEL_DIR / f.name
    shutil.copy(f, dst)
    print(f"  Saved: {f.name}")

# Save states
for f in STATE_DIR.glob('*'):
    dst = DRIVE_STATE_DIR / f.name
    if f.is_file():
        shutil.copy(f, dst)

# Save summary
summary = {
    'timestamp': datetime.now().isoformat(),
    'models': {}
}

for model_name in ENSEMBLE_CONFIGS.keys():
    state_file = STATE_DIR / f'{model_name}_pretrain_state.pth'
    done_file = STATE_DIR / f'{model_name}_pretrain_DONE'

    if state_file.exists():
        state = torch.load(state_file, map_location='cpu', weights_only=False)
        summary['models'][model_name] = {
            'epoch': state['epoch'],
            'best_loss': state['best_loss'],
            'complete': done_file.exists()
        }

with open(DRIVE_MODEL_DIR / 'pretrain_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nAll files saved to: {DRIVE_MODEL_DIR}")